# 01 YOLO Segmentation Training Only for AlphaDent

This notebook is a **training-only** notebook for the AlphaDent YOLO segmentation workflow.

## Current experiment (V13 — crop / tile-based training)

> ### ⚠️ V13 RESULT: FAILED (−0.11). V6 remains the best model.
>
> V13 has been trained and evaluated. On the **comparable** full-image (tiled + merged) metric it
> scored Mask mAP50-95 = **0.0993 vs V6's re-scored 0.2099 (−0.11)** — the worst result in the
> project (see `src/03` and `docs/AlphaDent_training_summary_EN.md`). Naive tiling clips the large
> objects out of training (`MIN_AREA_FRAC`), fragments them at inference, and the merge step never
> reassembles them, so the large classes that carry most of the per-class-averaged mAP collapse
> (Abrasion −0.41, Crown −0.43). **Use V6/V10 (≈0.234) for submissions.** This notebook is kept as
> the V13 reference implementation; the rationale below is the original (pre-result) hypothesis.

This is the **V13** experiment: `yolov8s-seg + tile-based training`. The notebook is
self-contained — running it top to bottom **slices the dataset into overlapping tiles
and then trains** on those tiles, all on Kaggle.

### Why tiling (original hypothesis — see the result banner above)

The V6 error analysis found ~78% of validation objects occupy <1% of the image area.
Every full-image lever has now plateaued at ~0.234 Mask mAP50-95: image size (V6–V8),
model size (V9), oversampling (V7/V10), augmentation (V11), and the **V12 P2 head**.
V12 is the decisive clue — adding a high-resolution detection head did **not** improve
recall, which means the bottleneck is the **full-image input**, not the head: a tiny
lesion is only a few pixels once the panoramic image is downscaled to 768.

V13 attacks that directly by changing the **input**: slice each panoramic image into
overlapping tiles and train on the tiles, so a lesion that was ~5 px in the full-image
input becomes ~20–40 px in a tile. The model finally gets enough pixels to learn a
fine-grained mask. This is the on-target fix the P2 head failed to deliver.

> **What actually happened:** tiling did enlarge the tiny lesions, but it also broke the *large*
> objects (clipped out of training, fragmented at inference, never reassembled). Since the large
> classes carry the mAP, the net effect was a large regression — see the V13 section in the
> experiment log for the full analysis and the "mAP weight ≠ object count" reframing.

### What changed vs the V6 baseline (single variable = tiled input)

1. **Tiled input**: the dataset is sliced into `TILE_SIZE`-px tiles with `OVERLAP`
   overlap; polygon labels are clipped + renormalized to each tile; object-free
   train tiles are subsampled. Training runs at `imgsz=TILE_SIZE` so each tile is fed
   ~1:1 (no extra downscaling).
2. **Stock head**: back to plain `yolov8s-seg` (V12's P2 head did not help).
3. **Augmentation = clean V6 baseline**: `mosaic=1.0`, `close_mosaic=10`, `mixup=0.0`,
   `copy_paste=0.0`. Oversampling stays off. Tiling is the only change.

### ⚠️ The reported val mAP is NOT comparable to the 0.234 baseline

Training validates on the **tiled** val split, which is an *easier* task (objects are
relatively larger), so `results.csv` Mask mAP will look inflated. Use it only for
early-stopping / monitoring. The number that is comparable to the ~0.234 full-image
baseline must come from **tiled + merged inference on the full val images**, computed
in `src/03-alphadent-val-map-eval.ipynb`.

### Result (the V13 questions, now answered)

Did tile-based training lift the *comparable* (full-image, merged) Mask mAP50-95 above the
~0.234 plateau, and — unlike P2 — did it finally improve **recall** on small lesions?
**No on both counts.** The comparable metric fell to 0.0993 (−0.11), and the tiny Caries
classes moved only ±0.01–0.02 while the large classes collapsed.

Outputs: `weights/best.pt` (use this), `weights/last.pt` (debug only), `results.csv`.


## 1. Environment Setup

This section imports the basic packages and sets a fixed random seed for reproducibility.

In [ ]:
import os
import random
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Current working directory:", os.getcwd())
print("Kaggle input directory exists:", Path("/kaggle/input").exists())
print("Kaggle working directory exists:", Path("/kaggle/working").exists())

In [ ]:
# Check GPU status.
# This is useful on Kaggle to confirm that the selected accelerator is available.
!nvidia-smi

## 2. Install and Import Ultralytics

If Ultralytics is already installed, this cell will finish quickly. If Internet is disabled, install Ultralytics from an uploaded Kaggle Dataset containing the required wheels.

In [ ]:
# Install Ultralytics if it is not already available.
try:
    import ultralytics
    print("Ultralytics is already installed.")
except ImportError:
    print("Installing Ultralytics...")
    !pip install -q ultralytics

In [ ]:
# Disable most progress-bar style logs before importing YOLO.
# This helps keep Kaggle notebook logs cleaner.
os.environ["YOLO_VERBOSE"] = "False"

from ultralytics import YOLO
from ultralytics.utils import LOGGER
import ultralytics

# Keep only important Ultralytics logs.
LOGGER.setLevel(logging.ERROR)

# Extra safeguard against verbose progress bars in some Ultralytics versions.
try:
    import ultralytics.utils as yolo_utils
    yolo_utils.VERBOSE = False
except Exception:
    pass

try:
    import ultralytics.utils.tqdm as yolo_tqdm
    yolo_tqdm.VERBOSE = False
except Exception:
    pass

print("Ultralytics version:", ultralytics.__version__)

## 3. Locate the Dataset YAML

This notebook searches for `yolo_seg_train.yaml` under `/kaggle/input` and uses it as the YOLO dataset configuration file.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

yaml_candidates = list(INPUT_ROOT.rglob("yolo_seg_train.yaml"))

print(f"Number of yolo_seg_train.yaml files found: {len(yaml_candidates)}")

for i, p in enumerate(yaml_candidates):
    print(f"[{i}] {p}")

if len(yaml_candidates) == 0:
    raise FileNotFoundError(
        "No yolo_seg_train.yaml found under /kaggle/input. "
        "Please check whether the AlphaDent dataset is added to this notebook."
    )

# If multiple YAML files are found, use the first one by default.
# Change YAML_INDEX manually if needed.
YAML_INDEX = 0
DATA_YAML = yaml_candidates[YAML_INDEX]

print("\nSelected DATA_YAML:", DATA_YAML)
print("Dataset root:", DATA_YAML.parent)

## 4. Inspect Dataset Metadata

This section reads the YAML file and checks the class names.

In [ ]:
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_yaml_dict = yaml.safe_load(f)

print("YAML content:")
for k, v in data_yaml_dict.items():
    print(f"{k}: {v}")

print("\nClass names:")
names = data_yaml_dict.get("names", None)
print(names)

if names is None:
    raise ValueError("The YAML file does not contain the 'names' field.")

if isinstance(names, dict):
    num_classes = len(names)
elif isinstance(names, list):
    num_classes = len(names)
else:
    raise TypeError("The 'names' field should be either a list or a dictionary.")

print("Number of classes:", num_classes)

## 5. Basic Path Check

This section resolves the train and validation image folders. The resolved paths are used to create a runtime YAML file with absolute paths for Ultralytics.

In [ ]:
dataset_root = DATA_YAML.parent

yaml_path_value = data_yaml_dict.get("path", None)

if yaml_path_value is not None:
    yaml_root = Path(yaml_path_value)
    if not yaml_root.is_absolute():
        yaml_root = dataset_root / yaml_root
else:
    yaml_root = dataset_root

print("Resolved YAML root:", yaml_root)

def resolve_split_path(split_value):
    # Resolve a split path from the YAML file.
    if split_value is None:
        return None

    split_path = Path(split_value)
    if split_path.is_absolute():
        return split_path

    # Try resolving relative to YAML root first.
    candidate_1 = yaml_root / split_path
    if candidate_1.exists():
        return candidate_1

    # Try resolving relative to the YAML file directory.
    candidate_2 = dataset_root / split_path
    if candidate_2.exists():
        return candidate_2

    return candidate_1

train_path = resolve_split_path(data_yaml_dict.get("train", None))
val_path = resolve_split_path(data_yaml_dict.get("val", data_yaml_dict.get("valid", None)))
test_path = resolve_split_path(data_yaml_dict.get("test", None))

print("Train path:", train_path, "| exists:", train_path.exists() if train_path else None)
print("Val path:", val_path, "| exists:", val_path.exists() if val_path else None)
print("Test path:", test_path, "| exists:", test_path.exists() if test_path else None)

if train_path is None or not train_path.exists():
    raise FileNotFoundError("Train path does not exist. Please check the YAML file.")

if val_path is None or not val_path.exists():
    raise FileNotFoundError("Validation path does not exist. Please check the YAML file.")

In [ ]:
# Count image files in each split.
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(folder):
    if folder is None or not folder.exists():
        return 0
    return sum(1 for p in folder.rglob("*") if p.suffix.lower() in image_exts)

split_counts = {
    "train": count_images(train_path),
    "val": count_images(val_path),
    "test": count_images(test_path),
}

pd.DataFrame(
    [{"split": k, "num_images": v} for k, v in split_counts.items()]
)

## 6. Training Configuration

Change the parameters in this section for different experiments. The `RUN_NAME` is generated automatically from the key training parameters so that the result folder name records the experiment setup.

In [ ]:
# =========================
# V13 Training configuration  (crop / tile-based training)
# =========================
# V12 (P2 head) confirmed the bottleneck is the FULL-IMAGE INPUT, not the detection
# head: tiny lesions are only a few pixels after a panoramic image is downscaled to
# 768. V13 changes the INPUT — slice each panoramic image into overlapping tiles and
# train on the tiles, so small lesions occupy many more pixels. The tiled input is the
# single variable vs the V6 baseline (stock yolov8s-seg, clean baseline augmentation).

MODEL_NAME = "yolov8s-seg.pt"      # stock head (V12's P2 head did not help)

EPOCHS = 120
BATCH_SIZE = 16
DEVICE = 0
WORKERS = 2
PATIENCE = 25

# AlphaDent classes are pathology types, not left/right positions, so horizontal
# flip stays off to avoid unrealistic dental layouts.
FLIPLR = 0.0

# --- Tiling parameters (mirror of tools/tile_yolo_seg.py) ---
TILE_SIZE = 640        # tile side in pixels; also the training imgsz
OVERLAP = 0.20         # fractional overlap between neighbouring tiles
KEEP_EMPTY = 0.15      # fraction of object-free TRAIN tiles to keep (val keeps all)
MIN_AREA_FRAC = 0.35   # keep a clipped object only if >= this share of its area survives

# Train at the tile size so a TILE_SIZE-px tile is fed ~1:1 (no extra downscaling).
IMG_SIZE = TILE_SIZE

# Clean V6 baseline augmentation: mosaic regularizes (V11 proved disabling it hurts),
# mixup/copy_paste off. Tiling is the only change vs the V6 baseline.
MOSAIC = 1.0
CLOSE_MOSAIC = 10
MIXUP = 0.0
COPY_PASTE = 0.0

TILES_DIR = Path("/kaggle/working/datasets/alphadent_tiles")
PROJECT_DIR = Path("/kaggle/working/alphadent_yolo")

# Automatically include key parameters in the run folder name.
model_tag = Path(MODEL_NAME).stem.replace("-", "_")
fliplr_tag = str(FLIPLR).replace(".", "p")
RUN_NAME = (
    f"v13_tile_{model_tag}_t{TILE_SIZE}_ov{int(OVERLAP*100)}_"
    f"bs{BATCH_SIZE}_ep{EPOCHS}_pat{PATIENCE}_fliplr{fliplr_tag}_seed{SEED}"
)

print("=== V13 tile-based training config ===")
print("Model:", MODEL_NAME)
print("Tile size / imgsz:", TILE_SIZE)
print("Overlap:", OVERLAP)
print("Keep-empty (train):", KEEP_EMPTY)
print("Min area frac:", MIN_AREA_FRAC)
print("Epochs:", EPOCHS, "| Batch:", BATCH_SIZE, "| Patience:", PATIENCE)
print("fliplr:", FLIPLR)
print("mosaic:", MOSAIC, "| close_mosaic:", CLOSE_MOSAIC, "| mixup:", MIXUP, "| copy_paste:", COPY_PASTE)
print("Tiles dir:", TILES_DIR)
print("Project dir:", PROJECT_DIR)
print("Run name:", RUN_NAME)

## 7. Build the Tiled Dataset

This section slices the original train/val images (and their polygon labels) into
overlapping tiles and writes a **new** YOLO-seg dataset under `/kaggle/working`.

It does **not** modify the original dataset. The tiling code below is an inline copy
of `tools/tile_yolo_seg.py` (kept in sync) so this notebook is self-contained on Kaggle.

For each tile:
- the image crop is saved,
- every polygon label is clipped to the tile and renormalized to the tile's frame,
- an object is dropped if less than `MIN_AREA_FRAC` of its area survives the clip
  (the overlapping neighbour tile keeps it instead),
- object-free **train** tiles are randomly subsampled to `KEEP_EMPTY` (so background
  tiles don't swamp the model); the **val** split keeps every tile for a stable,
  representative early-stopping signal.

Re-running this cell rebuilds the tile dataset from scratch (it is idempotent).</cell id="cell-16">

In [ ]:
# =========================
# V13: build the tiled dataset  (inline copy of tools/tile_yolo_seg.py)
# =========================
import shutil
import random
import cv2

IMG_EXTS_TILE = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def _starts(length, tile, step):
    if length <= tile:
        return [0]
    s = list(range(0, length - tile + 1, step))
    if s[-1] != length - tile:
        s.append(length - tile)
    return s


def compute_tiles(w, h, tile_size, overlap):
    step = max(1, int(round(tile_size * (1.0 - overlap))))
    boxes = []
    for y0 in _starts(h, tile_size, step):
        for x0 in _starts(w, tile_size, step):
            boxes.append((x0, y0, min(x0 + tile_size, w), min(y0 + tile_size, h)))
    return boxes


def _poly_area(pts):
    pts = np.asarray(pts, dtype=np.float64)
    if len(pts) < 3:
        return 0.0
    x, y = pts[:, 0], pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def clip_polygon_to_rect(poly, x0, y0, x1, y1):
    # Sutherland-Hodgman clip against the axis-aligned tile rectangle.
    def clip_edge(points, inside, intersect):
        out = []
        n = len(points)
        if n == 0:
            return out
        for i in range(n):
            cur, prev = points[i], points[i - 1]
            ci, pi = inside(cur), inside(prev)
            if ci:
                if not pi:
                    out.append(intersect(prev, cur))
                out.append(cur)
            elif pi:
                out.append(intersect(prev, cur))
        return out

    def iv(a, b, cx):
        dx = b[0] - a[0]
        t = 0.0 if dx == 0 else (cx - a[0]) / dx
        return (cx, a[1] + t * (b[1] - a[1]))

    def ih(a, b, cy):
        dy = b[1] - a[1]
        t = 0.0 if dy == 0 else (cy - a[1]) / dy
        return (a[0] + t * (b[0] - a[0]), cy)

    pts = [tuple(p) for p in poly]
    pts = clip_edge(pts, lambda p: p[0] >= x0, lambda a, b: iv(a, b, x0))
    pts = clip_edge(pts, lambda p: p[0] <= x1, lambda a, b: iv(a, b, x1))
    pts = clip_edge(pts, lambda p: p[1] >= y0, lambda a, b: ih(a, b, y0))
    pts = clip_edge(pts, lambda p: p[1] <= y1, lambda a, b: ih(a, b, y1))
    return pts


def parse_seg_label_line(line):
    parts = line.strip().split()
    if len(parts) < 7:
        return None
    try:
        cls = int(float(parts[0]))
        coords = [float(v) for v in parts[1:]]
    except ValueError:
        return None
    if len(coords) % 2 != 0:
        coords = coords[:-1]
    poly = np.asarray(coords, dtype=np.float64).reshape(-1, 2)
    if len(poly) < 3:
        return None
    return cls, poly


def tile_label_for_box(full_polys, box, w, h, min_area_frac):
    x0, y0, x1, y1 = box
    tw, th = x1 - x0, y1 - y0
    lines = []
    for cls, poly_norm in full_polys:
        p = poly_norm.copy()
        p[:, 0] *= w
        p[:, 1] *= h
        oa = _poly_area(p)
        if oa <= 0:
            continue
        c = clip_polygon_to_rect(p, x0, y0, x1, y1)
        if len(c) < 3:
            continue
        c = np.asarray(c, dtype=np.float64)
        if _poly_area(c) / oa < min_area_frac:
            continue
        c[:, 0] = (c[:, 0] - x0) / tw
        c[:, 1] = (c[:, 1] - y0) / th
        c = np.clip(c, 0.0, 1.0)
        lines.append(f"{cls} " + " ".join(f"{v:.6f}" for v in c.reshape(-1)))
    return lines


def _labels_dir_for(images_dir):
    parts = list(Path(images_dir).parts)
    if "images" in parts:
        idx = len(parts) - 1 - parts[::-1].index("images")
        parts[idx] = "labels"
        return Path(*parts)
    return Path(images_dir).parent / "labels"


def tile_split(src_images, out_images, out_labels, keep_empty, rng):
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)
    src_labels = _labels_dir_for(src_images)
    img_files = sorted(p for p in Path(src_images).iterdir() if p.suffix.lower() in IMG_EXTS_TILE)
    n_tiles = n_obj = 0
    for img_path in img_files:
        img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
        if img is None:
            continue
        h, w = img.shape[:2]
        lbl = src_labels / (img_path.stem + ".txt")
        full_polys = []
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parsed = parse_seg_label_line(line)
                if parsed is not None:
                    full_polys.append(parsed)
        for ti, box in enumerate(compute_tiles(w, h, TILE_SIZE, OVERLAP)):
            x0, y0, x1, y1 = box
            lines = tile_label_for_box(full_polys, box, w, h, MIN_AREA_FRAC)
            has_obj = len(lines) > 0
            if not has_obj and rng.random() > keep_empty:
                continue
            name = f"{img_path.stem}__t{ti}_{x0}_{y0}"
            cv2.imwrite(str(out_images / f"{name}.jpg"), img[y0:y1, x0:x1])
            (out_labels / f"{name}.txt").write_text(
                ("\n".join(lines) + "\n") if lines else "", encoding="utf-8"
            )
            n_tiles += 1
            n_obj += int(has_obj)
    return n_tiles, n_obj


# Build (idempotent: clear any previous tiles first so re-runs are clean).
if TILES_DIR.exists():
    shutil.rmtree(TILES_DIR)

rng = random.Random(SEED)
print("Tiling train split ...")
nt_tr, no_tr = tile_split(train_path, TILES_DIR / "images" / "train",
                          TILES_DIR / "labels" / "train", KEEP_EMPTY, rng)
print("Tiling val split (keeping all tiles) ...")
nt_va, no_va = tile_split(val_path, TILES_DIR / "images" / "val",
                          TILES_DIR / "labels" / "val", 1.0, rng)

print(f"\nTrain tiles : {nt_tr} ({no_tr} with objects, {nt_tr - no_tr} background)")
print(f"Val   tiles : {nt_va} ({no_va} with objects, {nt_va - no_va} background)")

TILES_YAML = TILES_DIR / "yolo_seg_tiles.yaml"
with open(TILES_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump({
        "path": str(TILES_DIR.resolve()),
        "train": "images/train",
        "val": "images/val",
        "nc": num_classes,
        "names": names,
    }, f, sort_keys=False, allow_unicode=True)

print("\nTiled dataset YAML written to:", TILES_YAML)

## 8. Point Training at the Tiled Dataset

The tiled dataset already has a self-contained YAML with an absolute `path`, so training
uses it directly. This replaces the original full-image dataset for V13.</cell id="cell-18">

In [ ]:
YOLO_DATA_YAML = TILES_YAML

print("Training will use the tiled dataset YAML:", YOLO_DATA_YAML)
print("-" * 60)
with open(YOLO_DATA_YAML, "r", encoding="utf-8") as f:
    print(f.read())
print("-" * 60)
print("\nREMINDER: the Mask mAP reported DURING training is computed on the TILED val")
print("split (an easier task) and is NOT directly comparable to the ~0.234 full-image")
print("baseline. Use it only for early stopping. The comparable number requires")
print("tiled + merged inference on the FULL val images (separate analysis notebook).")

## 9. Train YOLO Segmentation Model

V13 changes only the training **input** vs the clean V6 baseline:

- model: stock `yolov8s-seg.pt` (V12's P2 head did not help, so the head is reverted)
- data: the **tiled** dataset built above
- image size: `imgsz=TILE_SIZE` so each tile is fed ~1:1 (no extra downscaling)
- augmentation: V6/V10 baseline (`mosaic=1.0`, `close_mosaic=10`, `mixup=0.0`, `copy_paste=0.0`)
- oversampling: disabled

For later validation or submission, use `weights/best.pt` instead of `weights/last.pt`.</cell id="cell-20">

In [ ]:
# =========================
# V13: build the stock yolov8s-seg model and train on tiles
# =========================
# V12's P2 head did not lift the plateau and did not improve recall, so V13 reverts
# to the stock head. The only change vs the V6 baseline is the tiled input.
model = YOLO(MODEL_NAME)
print("Built stock yolov8s-seg model from:", MODEL_NAME)


# Custom concise logging callbacks.
# These callbacks print one message at the start of each epoch and one summary at the end.
def on_train_epoch_start(trainer):
    current_epoch = trainer.epoch + 1
    total_epochs = trainer.epochs
    print(f"\n========== Epoch {current_epoch}/{total_epochs} started ==========")


def on_fit_epoch_end(trainer):
    current_epoch = trainer.epoch + 1
    total_epochs = trainer.epochs

    print(f"---------- Epoch {current_epoch}/{total_epochs} finished ----------")

    # Print training losses if available.
    if hasattr(trainer, "tloss") and trainer.tloss is not None:
        try:
            loss_values = trainer.tloss.detach().cpu().numpy().tolist()
            loss_names = getattr(trainer, "loss_names", None)

            if loss_names is not None and len(loss_names) == len(loss_values):
                loss_text = ", ".join(
                    [f"{name}: {value:.4f}" for name, value in zip(loss_names, loss_values)]
                )
            else:
                loss_text = ", ".join(
                    [f"loss_{i}: {value:.4f}" for i, value in enumerate(loss_values)]
                )

            print("Train losses:", loss_text)
        except Exception as e:
            print("Train losses could not be printed:", e)

    # Print the main validation metrics recorded during training.
    if hasattr(trainer, "metrics") and trainer.metrics is not None:
        metrics_to_show = [
            "metrics/precision(B)",
            "metrics/recall(B)",
            "metrics/mAP50(B)",
            "metrics/mAP50-95(B)",
            "metrics/precision(M)",
            "metrics/recall(M)",
            "metrics/mAP50(M)",
            "metrics/mAP50-95(M)",
        ]

        metric_text_list = []
        for key in metrics_to_show:
            if key in trainer.metrics:
                metric_text_list.append(f"{key}: {trainer.metrics[key]:.4f}")

        if len(metric_text_list) > 0:
            print("Validation metrics (on TILED val — not comparable to the 0.234 baseline):")
            print(", ".join(metric_text_list))


model.add_callback("on_train_epoch_start", on_train_epoch_start)
model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

results = model.train(
    data=str(YOLO_DATA_YAML),
    task="segment",
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    seed=SEED,
    patience=PATIENCE,
    cache=False,
    pretrained=True,
    optimizer="auto",
    verbose=False,
    plots=False,
    fliplr=FLIPLR,
    mosaic=MOSAIC,
    close_mosaic=CLOSE_MOSAIC,
    mixup=MIXUP,
    copy_paste=COPY_PASTE,
)

# Store the actual output directory created by Ultralytics.
RUN_DIR = Path(model.trainer.save_dir)
WEIGHTS_DIR = RUN_DIR / "weights"
BEST_PT = WEIGHTS_DIR / "best.pt"
LAST_PT = WEIGHTS_DIR / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

print("\nActual Ultralytics run directory:", RUN_DIR)
print("Weights directory:", WEIGHTS_DIR)
print("best.pt exists:", BEST_PT.exists())
print("last.pt exists:", LAST_PT.exists())
print("results.csv exists:", RESULTS_CSV.exists())

print("\nImportant checkpoint note:")
print("Use best.pt for validation/submission. last.pt is kept only for comparison/debugging.")

## 10. Check Training Outputs

This cell checks that the required outputs were created.

The main checkpoint for later analysis and submission is:

- `weights/best.pt`

`weights/last.pt` is still saved, but it should not be used unless you explicitly want to compare it against `best.pt`.

In [ ]:
# Use the actual output directory from training if available.
# If the kernel was restarted, search for the latest matching run folder.

if "RUN_DIR" in globals() and Path(RUN_DIR).exists():
    RUN_DIR = Path(RUN_DIR)
else:
    candidate_run_dirs = sorted(
        [p for p in PROJECT_DIR.glob(f"{RUN_NAME}*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )

    print("Detected candidate run directories:")
    for p in candidate_run_dirs:
        print(p)

    RUN_DIR = None
    for p in candidate_run_dirs:
        if (p / "weights").exists():
            RUN_DIR = p
            break

    if RUN_DIR is None:
        raise FileNotFoundError("No YOLO run directory with a weights folder was found.")

WEIGHTS_DIR = RUN_DIR / "weights"
BEST_PT = WEIGHTS_DIR / "best.pt"
LAST_PT = WEIGHTS_DIR / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

required_outputs = {
    "best.pt": BEST_PT,
    "last.pt": LAST_PT,
    "results.csv": RESULTS_CSV,
}

print("Run directory:", RUN_DIR)
print("\nRequired training outputs:")

missing = []
for name, path in required_outputs.items():
    exists = path.exists()
    print(f"{name}: {path} | exists: {exists}")
    if not exists:
        missing.append(name)

if missing:
    raise FileNotFoundError(f"Missing required output files: {missing}")

print("\nTraining-only notebook completed successfully.")
print("Main checkpoint for the next notebook:", BEST_PT)
print("Use best.pt for error analysis and submission preparation.")

In [ ]:
# Optional: show only the last few rows of the training metrics CSV.
# This does not generate plots or image outputs.

metrics_df = pd.read_csv(RESULTS_CSV)
metrics_df.columns = [c.strip() for c in metrics_df.columns]
metrics_df.tail()

## Next Step

`best.pt` is trained on tiles. Two follow-ups:

1. **The comparable metric** — the Mask mAP in this run's `results.csv` is on the *tiled*
   val split and is **not** comparable to the ~0.234 full-image baseline. The comparable number
   comes from tiled + merged inference on the **full** val images, computed in
   `src/03-alphadent-val-map-eval.ipynb` (which also re-scores V6 with the same code).
2. **Submission** — `src/02` runs the tiled inference + merge pipeline. NB: see the result below
   before using a tiled model for submission.

### V13 result (these questions are now answered)

> Did tile-based training lift the *comparable* (full-image, merged) Mask mAP50-95 above the
> ~0.234 plateau? And — unlike the V12 P2 head — did it finally improve **recall** on the small
> Caries lesions?

**No.** V13 scored a comparable Mask mAP50-95 of **0.0993 vs V6's re-scored 0.2099 (−0.11)**, the
worst result in the project. Tiling fragments and discards the large objects (Abrasion, Crown,
Filling) that carry most of the per-class-averaged mAP, while the tiny Caries it was meant to help
moved only ±0.01–0.02. **V6/V10 (≈0.234) remain the best models and should be used for submissions.**
See `docs/AlphaDent_training_summary_EN.md` for the full analysis.

Do not judge this run by `last.pt`; use `best.pt`.
